# AI Usage Analyzer: Impact of AI on Students and Professionals
A menu-based Python tool that analyzes and visualizes how students and professionals use AI tools.

**Practicals used:** variables and operators, collections (list, tuple, string), sets and dictionaries, functions and files, **NumPy**, **Pandas**, **Matplotlib and Seaborn**, **web scraping** (Requests and BeautifulSoup).

**Dataset:** AI Usage and Impact on Students & Professionals (Kaggle, by Eisha Tuz Zuhra).

**Front end:** Step 3B builds an HTML dashboard (`ai_dashboard.html`) with a group dropdown, tabs, cards and charts, and opens it in your browser.

**How to run:** run the cells from top to bottom (Kernel > Restart Kernel and Run All Cells). The dashboard opens in the browser, and in the last cell a console menu appears: type a number and press Enter.

In [ ]:
import io
import os
import re
import html
import base64
import warnings
import webbrowser
import contextlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from bs4 import BeautifulSoup

warnings.filterwarnings("ignore")          # hide small warning messages
sns.set_theme(style="whitegrid")

In [ ]:
# Step 1: get the dataset file into the notebook
CSV_PATH = "ai_usage.csv"      # tip: rename the downloaded Kaggle file to ai_usage.csv

try:
    from google.colab import files      # works only on Google Colab
    if not os.path.exists(CSV_PATH):
        print("Choose the AI usage CSV file from your computer...")
        files.upload()
except ImportError:
    pass   # Jupyter on your PC: keep the CSV in the same folder as this notebook

# if the file still has its original name, find it (any .csv with "usage" in the name)
if not os.path.exists(CSV_PATH):
    found = [f for f in os.listdir() if f.lower().endswith(".csv") and "usage" in f.lower()]
    if found:
        CSV_PATH = found[0]

print("Working folder:", os.getcwd())
print("CSV files in this folder:", [f for f in os.listdir() if f.lower().endswith(".csv")])
print("Using file:", CSV_PATH)
print("File found:", os.path.exists(CSV_PATH))
if not os.path.exists(CSV_PATH):
    print("PROBLEM: the CSV is not in the working folder. Copy it next to this notebook and run this cell again.")

In [ ]:
# Step 2: load and clean the data (Pandas)
EXPECTED_COLUMNS = ["User_ID", "Age", "Gender", "Country", "User_Type", "Education_Level", "Profession",
                    "Monthly_Income", "AI_Tool", "AI_Purpose", "AI_Usage_Hours_Per_Day", "Tasks_Performed",
                    "Monthly_AI_Cost", "Productivity_Score", "Accuracy_Rating", "Satisfaction_Score",
                    "Time_Saved_Hours_Per_Week", "Work_or_Study_Hours_Per_Day", "Would_Recommend"]   # list
NUMBER_COLUMNS = ["Age", "Monthly_Income", "AI_Usage_Hours_Per_Day", "Monthly_AI_Cost", "Productivity_Score",
                  "Accuracy_Rating", "Satisfaction_Score", "Time_Saved_Hours_Per_Week", "Work_or_Study_Hours_Per_Day"]
TEXT_COLUMNS = ["Gender", "Country", "User_Type", "Education_Level", "Profession", "AI_Tool", "AI_Purpose"]
YES_VALUES = {"yes", "y", "true", "1", "1.0"}       # set: the answers that mean "yes"

def load_data(path):
    """Read the CSV file and prepare it for analysis."""
    df = pd.read_csv(path)                                    # file reading
    df.columns = df.columns.str.strip()                       # remove spaces from column names

    # make the column names match exactly (capital or small letters do not matter)
    proper = {name.lower(): name for name in EXPECTED_COLUMNS}                       # dictionary
    df = df.rename(columns={c: proper[c.lower()] for c in df.columns if c.lower() in proper})
    missing = [c for c in EXPECTED_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"These columns are missing in the CSV: {missing}. Columns found: {list(df.columns)}")

    for col in NUMBER_COLUMNS:
        df[col] = pd.to_numeric(df[col], errors="coerce")     # text such as "NA" becomes NaN
    for col in TEXT_COLUMNS:
        df[col] = df[col].astype(str).str.strip().replace({"nan": np.nan, "": np.nan})   # string cleaning

    # two new columns made from existing ones
    df["Recommend_Flag"] = df["Would_Recommend"].astype(str).str.strip().str.lower().isin(YES_VALUES)
    df["Plan"] = np.where(df["Monthly_AI_Cost"] > 0, "Paid", "Free")
    df.loc[df["Monthly_AI_Cost"].isnull(), "Plan"] = np.nan
    return df

df = load_data(CSV_PATH)
groups = ["All"] + sorted(set(df["User_Type"].dropna()))     # a set removes duplicates, sorted() makes a list

print("Total rows:", len(df))
print("Users by type:", df["User_Type"].value_counts().to_dict())
print("Countries:", df["Country"].nunique(), "| AI tools:", df["AI_Tool"].nunique())
print("Missing values:", int(df[EXPECTED_COLUMNS].isnull().sum().sum()))
print("STEP 2 DONE: the data is ready, now run Step 3 and Step 4.")

In [ ]:
# Step 3: constants and the nine features
METRICS = {"AI_Usage_Hours_Per_Day": "AI usage (hours per day)",
           "Productivity_Score": "Productivity score",
           "Accuracy_Rating": "Accuracy rating",
           "Satisfaction_Score": "Satisfaction score",
           "Time_Saved_Hours_Per_Week": "Time saved (hours per week)",
           "Monthly_AI_Cost": "Monthly AI cost"}                                  # dictionary: column -> nice name
RATING_COLS = ("Satisfaction_Score", "Accuracy_Rating", "Productivity_Score")      # tuple (fixed list)

def choose_group(prompt="Group (press Enter for All): "):
    print("Groups:", ", ".join(groups))
    while True:
        name = input(prompt).strip().lower()
        if name == "":
            return "All"
        exact = [g for g in groups if g.lower() == name]
        if exact:
            return exact[0]
        partial = [g for g in groups if name in g.lower()]
        if len(partial) == 1:
            return partial[0]
        print("Group not found, please try again.")

def get_group_data(group):
    """Boolean filter: keep only the rows of one group."""
    if group == "All":
        return df
    return df[df["User_Type"] == group]

# Feature 1: summary statistics (NumPy)
def group_summary(group):
    group_df = get_group_data(group)
    print(f"\n--- AI usage summary: {group} ---")
    print(f"Respondents    : {len(group_df)}")
    print(f"Average age    : {group_df['Age'].mean():.1f} years")
    for col, name in METRICS.items():
        values = group_df[col].dropna().to_numpy()        # NumPy array without missing values
        if values.size == 0:
            continue
        print(f"{name:<27}: mean {np.mean(values):.2f} | median {np.median(values):.2f} | "
              f"std {np.std(values):.2f} | max {np.max(values):.2f}")
    print(f"Would recommend: {group_df['Recommend_Flag'].mean() * 100:.1f}%")
    tools = group_df["AI_Tool"].value_counts()
    if not tools.empty:
        print(f"Most used tool : {tools.idxmax()}")

# Feature 2: students vs professionals (Seaborn bar chart)
def compare_user_types():
    cols = ["AI_Usage_Hours_Per_Day", "Productivity_Score", "Satisfaction_Score",
            "Accuracy_Rating", "Time_Saved_Hours_Per_Week"]
    averages = df.groupby("User_Type")[cols].mean().reset_index()
    print("\n--- Average values by user type ---")
    print(averages.round(2).to_string(index=False))
    long_form = averages.melt(id_vars="User_Type", var_name="Measure", value_name="Average")
    long_form["Measure"] = long_form["Measure"].map(METRICS)          # column name -> nice name
    plt.figure(figsize=(9, 5))
    sns.barplot(data=long_form, x="Measure", y="Average", hue="User_Type")
    plt.xticks(rotation=20, ha="right")
    plt.title("Students vs professionals")
    plt.xlabel("")
    plt.show()

# Feature 3: most used AI tools (Seaborn count plot)
def tool_popularity(group):
    group_df = get_group_data(group)
    counts = group_df["AI_Tool"].value_counts()
    if counts.empty:
        print("No AI tool data available for", group)
        return
    print(f"\n--- Most used AI tools: {group} ---")
    for tool, number in counts.items():
        print(f"{tool:<15}: {number} users ({number / counts.sum() * 100:.1f}%)")
    plt.figure(figsize=(8, 5))
    if group == "All":
        sns.countplot(data=group_df, y="AI_Tool", order=counts.index, hue="User_Type")
    else:
        sns.countplot(data=group_df, y="AI_Tool", order=counts.index, color="steelblue")
    plt.title(f"Most used AI tools - {group}")
    plt.xlabel("Number of users")
    plt.ylabel("AI tool")
    plt.show()

# Feature 4: daily AI use vs time saved (NumPy correlation + Seaborn)
def usage_vs_time_saved(group):
    pair = get_group_data(group)[["AI_Usage_Hours_Per_Day", "Time_Saved_Hours_Per_Week"]].dropna()
    if len(pair) < 3:
        print("Not enough data for", group)
        return
    x = pair["AI_Usage_Hours_Per_Day"].to_numpy()
    y = pair["Time_Saved_Hours_Per_Week"].to_numpy()
    r = np.corrcoef(x, y)[0, 1]                      # correlation value between -1 and +1
    slope, intercept = np.polyfit(x, y, 1)           # best straight line: y = slope*x + intercept
    print(f"\n--- AI use per day vs time saved per week: {group} ---")
    print(f"Correlation (r): {r:.3f}")
    print(f"Slope          : {slope:+.2f} hours saved per week for each extra hour of daily AI use")
    if abs(r) >= 0.7:
        print("Meaning        : strong relationship")
    elif abs(r) >= 0.4:
        print("Meaning        : moderate relationship")
    else:
        print("Meaning        : weak relationship")
    plt.figure(figsize=(7, 5))
    sns.regplot(data=pair, x="AI_Usage_Hours_Per_Day", y="Time_Saved_Hours_Per_Week",
                scatter_kws={"s": 18, "alpha": 0.5}, line_kws={"color": "red"})
    plt.title(f"AI use vs time saved - {group}")
    plt.xlabel("AI usage (hours per day)")
    plt.ylabel("Time saved (hours per week)")
    plt.show()

# Feature 5: which AI tool is rated best? (Pandas groupby + Seaborn)
def tool_ratings(group):
    group_df = get_group_data(group)
    ratings = group_df.groupby("AI_Tool")[list(RATING_COLS)].mean()
    if ratings.empty:
        print("No rating data available for", group)
        return
    ratings["Users"] = group_df.groupby("AI_Tool").size()
    ratings = ratings.sort_values("Satisfaction_Score", ascending=False)
    print(f"\n--- Average ratings of each AI tool: {group} ---")
    print(ratings.round(2).to_string())
    print("Highest satisfaction:", ratings.index[0])
    long_form = ratings[list(RATING_COLS)].reset_index().melt(
        id_vars="AI_Tool", var_name="Rating", value_name="Average score")
    plt.figure(figsize=(9, 5))
    sns.barplot(data=long_form, x="AI_Tool", y="Average score", hue="Rating")
    plt.xticks(rotation=20, ha="right")
    plt.title(f"Ratings of AI tools - {group}")
    plt.xlabel("")
    plt.show()

# Feature 6: why do people use AI? (Matplotlib pie chart)
def purpose_chart(group):
    counts = get_group_data(group)["AI_Purpose"].value_counts()
    if counts.empty:
        print("No purpose data available for", group)
        return
    if len(counts) > 7:                                   # keep the 7 biggest, group the rest as Other
        counts = pd.concat([counts.head(7), pd.Series({"Other": counts.iloc[7:].sum()})])
    print(f"\n--- Purpose of AI use: {group} ---")
    for name, number in counts.items():
        print(f"{name:<18}: {int(number)} ({number / counts.sum() * 100:.1f}%)")
    plt.figure(figsize=(7, 7))
    plt.pie(counts.values, labels=counts.index, autopct="%1.1f%%", startangle=90,
            textprops={"fontsize": 8})
    plt.title(f"Why people use AI - {group}")
    plt.show()

# Feature 7: paid vs free AI users (Pandas + Seaborn)
def paid_vs_free(group):
    group_df = get_group_data(group).dropna(subset=["Plan"])
    cols = ["Productivity_Score", "Satisfaction_Score", "Time_Saved_Hours_Per_Week"]
    if group_df.empty:
        print("No cost data available for", group)
        return
    averages = group_df.groupby("Plan")[cols].mean()
    print(f"\n--- Paid vs free AI users: {group} ---")
    print("Users:", group_df["Plan"].value_counts().to_dict())
    print(averages.round(2).to_string())
    long_form = averages.reset_index().melt(id_vars="Plan", var_name="Measure", value_name="Average")
    long_form["Measure"] = long_form["Measure"].map(METRICS)
    plt.figure(figsize=(8, 4))
    sns.barplot(data=long_form, x="Measure", y="Average", hue="Plan")
    plt.title(f"Paid vs free users - {group}")
    plt.xlabel("")
    plt.show()

# Feature 8: web scraping - country population vs AI use (Requests + BeautifulSoup)
POP_URL = "https://en.wikipedia.org/wiki/List_of_countries_and_dependencies_by_population"
COUNTRY_ALIASES = {"usa": "united states", "us": "united states", "u.s.": "united states",
                   "united states of america": "united states", "uk": "united kingdom",
                   "u.k.": "united kingdom", "england": "united kingdom", "uae": "united arab emirates",
                   "russian federation": "russia", "korea": "south korea"}      # dictionary: other names

def clean_name(text):
    text = re.sub(r"\[.*?\]", "", str(text))              # remove footnotes like [1]
    return text.lower().replace("&", "and").strip()

def parse_population_table(page_text):
    """Read the HTML of the page and return a dictionary {country: population}."""
    soup = BeautifulSoup(page_text, "html.parser")
    for table in soup.find_all("table", class_="wikitable"):
        rows = table.find_all("tr")
        if not rows:
            continue
        headers = [cell.get_text(strip=True).lower() for cell in rows[0].find_all(["th", "td"])]
        name_col = next((i for i, h in enumerate(headers)
                         if "country" in h or "location" in h or "territory" in h), None)
        pop_col = next((i for i, h in enumerate(headers) if "population" in h), None)
        if name_col is None or pop_col is None:
            continue                                      # not the table we want, try the next one
        population = {}
        for row in rows[1:]:
            cells = [c.get_text(strip=True) for c in row.find_all(["th", "td"])]
            if len(cells) <= max(name_col, pop_col):
                continue
            number = re.sub(r"\[.*?\]", "", cells[pop_col]).split("(")[0]
            number = re.sub(r"[^0-9]", "", number)        # keep digits only
            if number:
                population[clean_name(cells[name_col])] = int(number)
        return population
    return {}

def scrape_population(url=POP_URL):
    response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=10)
    response.raise_for_status()
    return parse_population_table(response.text)

def country_vs_population():
    print("Scraping country populations from the web...")
    try:
        population = scrape_population()
    except requests.RequestException as error:
        print("Could not reach the website. Check your internet connection.")
        print("Details:", error)
        return
    stats = df.groupby("Country")["AI_Usage_Hours_Per_Day"].agg(["mean", "count"]).reset_index()
    rows = []
    for _, row in stats.iterrows():
        key = clean_name(row["Country"])
        key = COUNTRY_ALIASES.get(key, key)
        if key in population and not pd.isnull(row["mean"]):
            rows.append({"Country": row["Country"], "Population": population[key],
                         "Respondents": int(row["count"]),
                         "Avg AI use (hours/day)": round(row["mean"], 2)})
    if len(rows) < 3:
        print("Not enough matching countries found (the web page layout may have changed).")
        return
    result = pd.DataFrame(rows).sort_values("Population", ascending=False)
    print(f"\nMatched {len(result)} countries from the web and our dataset:\n")
    print(result.to_string(index=False))
    r = result["Population"].corr(result["Avg AI use (hours/day)"])
    print(f"\nCorrelation between population and average AI use: {r:.2f}")
    plt.figure(figsize=(8, 5))
    sns.scatterplot(data=result, x="Population", y="Avg AI use (hours/day)", s=80)
    plt.xscale("log")
    for _, row in result.iterrows():
        plt.annotate(row["Country"], (row["Population"], row["Avg AI use (hours/day)"]), fontsize=8)
    plt.title("Country population (scraped) vs average AI use")
    plt.show()

# Feature 9: HTML report that opens in the browser (no backend needed)
def generate_report(group):
    sections = []   # each item: (title, printed text, chart image)
    images = []

    def capture_chart():
        # used instead of plt.show(): saves the chart as text (base64 PNG) for the page
        buffer = io.BytesIO()
        plt.savefig(buffer, format="png", bbox_inches="tight")
        plt.close()
        images.append(base64.b64encode(buffer.getvalue()).decode())

    steps = [("Summary", group_summary),
             ("Most used AI tools", tool_popularity),
             ("AI use vs time saved", usage_vs_time_saved),
             ("Ratings of AI tools", tool_ratings),
             ("Why people use AI", purpose_chart),
             ("Paid vs free users", paid_vs_free)]

    real_show = plt.show
    plt.show = capture_chart
    try:
        for title, func in steps:
            text = io.StringIO()
            before = len(images)
            with contextlib.redirect_stdout(text):        # capture print() output
                func(group)
            image = images[-1] if len(images) > before else None
            sections.append((title, text.getvalue(), image))
    finally:
        plt.show = real_show

    parts = []
    for title, text, image in sections:
        parts.append("<h2>" + title + "</h2>")
        parts.append("<pre>" + html.escape(text) + "</pre>")
        if image:
            parts.append('<img src="data:image/png;base64,' + image + '">')

    style = "body{font-family:Arial;max-width:800px;margin:auto;padding:20px} img{max-width:100%} pre{background:#f4f4f4;padding:10px}"
    page = ("<html><head><meta charset='utf-8'><title>AI Usage Analyzer - " + group + "</title><style>"
            + style + "</style></head><body><h1>AI Usage Analyzer report: " + group + "</h1>"
            + "".join(parts) + "</body></html>")

    filename = "report_" + group.replace(" ", "_") + ".html"
    with open(filename, "w", encoding="utf-8") as f:        # file writing
        f.write(page)
    print("Report saved as", filename)
    try:
        webbrowser.open("file://" + os.path.abspath(filename))
    except Exception:
        pass

In [ ]:
# Step 3B: front end - an HTML dashboard with a dropdown, tabs and charts (HTML + CSS + JavaScript)
import json

FEATURE_LIST = [("summary", "Summary", True), ("compare", "Students vs professionals", False),
                ("tools", "Most used tools", True), ("usage", "AI use vs time saved", True),
                ("ratings", "Tool ratings", True), ("purpose", "Why people use AI", True),
                ("paid", "Paid vs free", True), ("scrape", "Web scraping", False)]     # list of tuples

def capture(func, *args):
    """Run one feature and return its printed text and its chart (as base64 text)."""
    images = []

    def save_chart():
        buffer = io.BytesIO()
        plt.savefig(buffer, format="png", bbox_inches="tight", dpi=80)
        plt.close()
        images.append(base64.b64encode(buffer.getvalue()).decode())

    text = io.StringIO()
    real_show = plt.show
    plt.show = save_chart
    try:
        with contextlib.redirect_stdout(text):
            func(*args)
    except Exception as error:
        text.write(f"\nCould not create this view: {error}")
    finally:
        plt.show = real_show
        plt.close("all")
    return text.getvalue().strip(), (images[-1] if images else None)

def number_text(value, decimals=1):
    return "n/a" if pd.isnull(value) else f"{value:.{decimals}f}"

def make_cards(group):
    """The five numbers shown at the top of the dashboard."""
    group_df = get_group_data(group)
    return [["Respondents", str(len(group_df))],
            ["AI use (hours/day)", number_text(group_df["AI_Usage_Hours_Per_Day"].mean())],
            ["Time saved (hours/week)", number_text(group_df["Time_Saved_Hours_Per_Week"].mean())],
            ["Productivity score", number_text(group_df["Productivity_Score"].mean())],
            ["Would recommend", number_text(group_df["Recommend_Flag"].mean() * 100) + "%"]]

TEMPLATE = """<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>AI Usage Analyzer - Dashboard</title>
<style>
:root { --navy: #1f3a5f; --teal: #2a7f8e; --light: #eaf2f5; --grey: #f4f6f8; }
* { box-sizing: border-box; }
body { margin: 0; font-family: "Segoe UI", Arial, sans-serif; background: var(--grey); color: #1c2733; }
header { background: linear-gradient(90deg, var(--navy), var(--teal)); color: #fff; padding: 18px 28px; }
header h1 { margin: 0; font-size: 24px; }
header p { margin: 4px 0 0; opacity: 0.85; font-size: 14px; }
main { max-width: 1100px; margin: 0 auto; padding: 20px; }
.controls { display: flex; gap: 12px; align-items: center; flex-wrap: wrap; margin-bottom: 14px; }
select { padding: 8px 12px; border-radius: 8px; border: 1px solid #b8c4cc; font-size: 15px; background: #fff; }
.cards { display: grid; grid-template-columns: repeat(auto-fit, minmax(170px, 1fr)); gap: 12px; margin-bottom: 16px; }
.card { background: #fff; border-radius: 12px; padding: 14px; box-shadow: 0 1px 4px rgba(0, 0, 0, 0.08); }
.card .value { font-size: 26px; font-weight: 700; color: var(--navy); }
.card .label { font-size: 13px; color: #5a6a78; }
.tabs { display: flex; gap: 8px; flex-wrap: wrap; margin-bottom: 14px; }
.tab { padding: 8px 14px; border: none; border-radius: 20px; background: #fff; cursor: pointer; font-size: 14px; box-shadow: 0 1px 3px rgba(0, 0, 0, 0.12); }
.tab:hover { background: var(--light); }
.tab.active { background: var(--navy); color: #fff; }
.panel { background: #fff; border-radius: 12px; padding: 18px; box-shadow: 0 1px 4px rgba(0, 0, 0, 0.08); }
.panel h2 { margin-top: 0; color: var(--navy); }
.panel img { max-width: 100%; display: block; margin: 0 auto 14px; }
pre { background: var(--light); padding: 12px; border-radius: 8px; overflow: auto; font-size: 13px; white-space: pre-wrap; }
.note { font-size: 13px; color: #5a6a78; margin: 0 0 10px; }
footer { text-align: center; color: #5a6a78; font-size: 12px; padding: 16px; }
</style>
</head>
<body>
<header>
  <h1>AI Usage Analyzer</h1>
  <p>Impact of AI on students and professionals | Python (Pandas, NumPy, Seaborn) + HTML, CSS and JavaScript</p>
</header>
<main>
  <div class="controls">
    <label for="group"><b>Group:</b></label>
    <select id="group"></select>
  </div>
  <div class="cards" id="cards"></div>
  <div class="tabs" id="tabs"></div>
  <div class="panel">
    <h2 id="title"></h2>
    <p class="note" id="note"></p>
    <img id="chart" alt="chart">
    <pre id="text"></pre>
  </div>
</main>
<footer>Data: AI Usage and Impact on Students and Professionals (Kaggle). Charts made with Python.</footer>
<script>
const DATA = __DATA__;
let current = DATA.features[0].key;
const groupSelect = document.getElementById("group");

function renderCards() {
  const box = document.getElementById("cards");
  box.innerHTML = "";
  DATA.cards[groupSelect.value].forEach(function (item) {
    const card = document.createElement("div");
    card.className = "card";
    const value = document.createElement("div");
    value.className = "value";
    value.textContent = item[1];
    const label = document.createElement("div");
    label.className = "label";
    label.textContent = item[0];
    card.appendChild(value);
    card.appendChild(label);
    box.appendChild(card);
  });
}

function renderPanel() {
  const feature = DATA.features.find(function (f) { return f.key === current; });
  const entry = DATA.content[current][feature.per_group ? groupSelect.value : "*"];
  document.getElementById("title").textContent = feature.label + (feature.per_group ? " - " + groupSelect.value : "");
  document.getElementById("note").textContent = feature.per_group ? "" : "This view always uses all users, so the group box does not change it.";
  const img = document.getElementById("chart");
  if (entry.image) {
    img.src = "data:image/png;base64," + entry.image;
    img.style.display = "block";
  } else {
    img.removeAttribute("src");
    img.style.display = "none";
  }
  document.getElementById("text").textContent = entry.text || "No text output for this view.";
  document.querySelectorAll(".tab").forEach(function (button) {
    button.classList.toggle("active", button.dataset.key === current);
  });
}

DATA.groups.forEach(function (name) {
  const option = document.createElement("option");
  option.value = name;
  option.textContent = name;
  groupSelect.appendChild(option);
});
DATA.features.forEach(function (feature) {
  const button = document.createElement("button");
  button.className = "tab";
  button.dataset.key = feature.key;
  button.textContent = feature.label;
  button.addEventListener("click", function () { current = feature.key; renderPanel(); });
  document.getElementById("tabs").appendChild(button);
});
groupSelect.addEventListener("change", function () { renderCards(); renderPanel(); });
renderCards();
renderPanel();
</script>
</body>
</html>"""

def build_dashboard(filename="ai_dashboard.html"):
    print("Building the dashboard, please wait a few seconds...")
    per_group = {"summary": group_summary, "tools": tool_popularity, "usage": usage_vs_time_saved,
                 "ratings": tool_ratings, "purpose": purpose_chart, "paid": paid_vs_free}   # dictionary of functions
    content = {}
    for key, func in per_group.items():
        content[key] = {}
        for group in groups:
            text, image = capture(func, group)
            content[key][group] = {"text": text, "image": image}
    for key, func in [("compare", compare_user_types), ("scrape", country_vs_population)]:
        text, image = capture(func)
        content[key] = {"*": {"text": text, "image": image}}      # "*" means: the same for every group

    data = {"groups": groups,
            "features": [{"key": k, "label": label, "per_group": flag} for k, label, flag in FEATURE_LIST],
            "cards": {group: make_cards(group) for group in groups},
            "content": content}
    page = TEMPLATE.replace("__DATA__", json.dumps(data).replace("</", "<\\/"))
    with open(filename, "w", encoding="utf-8") as f:           # file writing
        f.write(page)
    print("Dashboard saved as", filename)
    try:
        webbrowser.open("file://" + os.path.abspath(filename))
    except Exception:
        pass

# build the dashboard now (it opens in your browser)
try:
    build_dashboard()
except Exception as error:
    print("The dashboard could not be built:", error)

In [ ]:
# Step 4: the menu (run this cell to start the program)
def main_menu():
    if "groups" not in globals():
        print("The data is not loaded. Run Step 2 first (Kernel > Restart Kernel and Run All Cells).")
        return
    while True:
        print("\n===== AI Usage Analyzer =====")
        print("1. Summary statistics")
        print("2. Students vs professionals")
        print("3. Most used AI tools")
        print("4. AI use per day vs time saved")
        print("5. Ratings of each AI tool")
        print("6. Why people use AI (pie chart)")
        print("7. Paid vs free AI users")
        print("8. Web scraping: country population vs AI use")
        print("9. Generate web (HTML) report")
        print("10. Front end: build and open the dashboard")
        print("0. Exit")
        choice = input("Enter your choice (0-10): ").strip()

        if choice == "0":
            print("Thank you for using AI Usage Analyzer!")
            break
        try:
            if choice == "1":
                group_summary(choose_group())
            elif choice == "2":
                compare_user_types()
            elif choice == "3":
                tool_popularity(choose_group())
            elif choice == "4":
                usage_vs_time_saved(choose_group())
            elif choice == "5":
                tool_ratings(choose_group())
            elif choice == "6":
                purpose_chart(choose_group())
            elif choice == "7":
                paid_vs_free(choose_group())
            elif choice == "8":
                country_vs_population()
            elif choice == "9":
                generate_report(choose_group())
            elif choice == "10":
                build_dashboard()
            else:
                print("Invalid choice, please enter a number from 0 to 10.")
        except Exception as error:      # the menu keeps running even if one option fails
            print("Something went wrong in this option:", error)

main_menu()